# Naive Bayes Spam Classifier from Scratch

Binary text classification using Naive Bayes — implemented without any ML libraries.
Covers tokenization, bag-of-words, prior/conditional probabilities, the zero-probability problem, and Laplace smoothing.

In [1]:
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist

# Labelled training corpus
spam = [
    "free laser hair treatment will improve your dating prospects",
    "leverage this weird blockchain hack to generate free cash",
    "new blockchain dating app needs single men",
    "change your life prospects with a single liposuction treatment",
    "one weird trick for free real estate"
]
notspam = [
    "can we reschedule for next week",
    "i will be back in the office on monday",
    "thanks for all your hard work this week",
    "i like your proposal in principle but will need to see more detail",
    "will you be free at 3 pm"
]
unlabelled = "blockchain proposal"

## Step 1: Tokenization

In [2]:
# Split each sentence into a list of tokens
spam_tokenized    = [word_tokenize(sent) for sent in spam]
notspam_tokenized = [word_tokenize(sent) for sent in notspam]

print("Spam tokens:", spam_tokenized[:2])
print("Not-spam tokens:", notspam_tokenized[:2])

Spam tokens: [['free', 'laser', 'hair', 'treatment', 'will', 'improve', 'your', 'dating', 'prospects'], ['leverage', 'this', 'weird', 'blockchain', 'hack', 'to', 'generate', 'free', 'cash']]
Not-spam tokens: [['can', 'we', 'reschedule', 'for', 'next', 'week'], ['i', 'will', 'be', 'back', 'in', 'the', 'office', 'on', 'monday']]


## Step 2: Training Dataset

In [3]:
# Pair each tokenized sentence with its class label: 1=spam, 0=not-spam
training_data = [(tokens, 0) for tokens in notspam_tokenized] + [(tokens, 1) for tokens in spam_tokenized]

## Step 3: Bag-of-Words Representation

In [4]:
# Convert each token list to a FreqDist — a count of each unique token
training_bag = [(FreqDist(tokens), label) for tokens, label in training_data]

## Step 4: Prior Probabilities

$$P(c) = \frac{\text{freq}(c)}{N}$$

How often each class appears in the training data.

In [5]:
total_count   = len(training_data)
spam_count    = sum(1 for _, label in training_data if label == 1)
notspam_count = total_count - spam_count

# class_priors[0] = P(not-spam), class_priors[1] = P(spam)
class_priors = [notspam_count / total_count, spam_count / total_count]
print("P(not-spam):", class_priors[0], " P(spam):", class_priors[1])

P(not-spam): 0.5  P(spam): 0.5


## Step 5: Conditional Probabilities

$$P(f|c) = \frac{\text{freq}(f,c)}{\text{freq}(c)}$$

How often each token appears in documents of each class.

In [6]:
cond_probs = {}
for bag, label in training_bag:
    for token in bag:
        counts = cond_probs.get(token, [0, 0])
        counts[label] += 1
        cond_probs[token] = counts

# Normalise by class frequency to get probabilities
for token in cond_probs:
    cond_probs[token][0] /= notspam_count
    cond_probs[token][1] /= spam_count

# Inspect a few tokens
for t in ['free', 'blockchain', 'proposal', 'will']:
    if t in cond_probs:
        print(f"P({t:<12} | not-spam) = {cond_probs[t][0]:.2f}   P({t:<12} | spam) = {cond_probs[t][1]:.2f}")

P(free         | not-spam) = 0.20   P(free         | spam) = 0.60
P(blockchain   | not-spam) = 0.00   P(blockchain   | spam) = 0.40
P(proposal     | not-spam) = 0.20   P(proposal     | spam) = 0.00
P(will         | not-spam) = 0.60   P(will         | spam) = 0.20


## Step 6: Prediction — The Zero-Probability Problem

Naive Bayes multiplies priors by conditional probabilities across all tokens.
If any token has probability 0 for a class, the entire product collapses to 0.

In [7]:
unlabelled_tokens = word_tokenize(unlabelled)

scores = list(class_priors)  # start with priors
for token in unlabelled_tokens:
    probs = cond_probs.get(token, [0, 0])
    scores[0] *= probs[0]   # P(not-spam) contribution
    scores[1] *= probs[1]   # P(spam) contribution
    print(f"Token '{token}': P(not-spam)={probs[0]:.4f}  P(spam)={probs[1]:.4f}")

print(f"\nFinal scores: not-spam={scores[0]:.6f}  spam={scores[1]:.6f}")
print("Result:", "Not Spam" if scores[0] >= scores[1] else "Spam")
print("\n→ Both scores are 0.0: 'blockchain' has 0 prob for not-spam, 'proposal' has 0 for spam.")

Token 'blockchain': P(not-spam)=0.0000  P(spam)=0.4000
Token 'proposal': P(not-spam)=0.2000  P(spam)=0.0000

Final scores: not-spam=0.000000  spam=0.000000
Result: Not Spam

→ Both scores are 0.0: 'blockchain' has 0 prob for not-spam, 'proposal' has 0 for spam.


## Step 7: Laplace (Add-One) Smoothing

To fix zero probabilities, add 1 to every count and add vocabulary size V to the denominator:

$$P(f|c) = \frac{\text{freq}(f,c) + 1}{\text{freq}(c) + V}$$

No probability is ever exactly zero after smoothing.

In [8]:
V = len(cond_probs)  # vocabulary size

for token in cond_probs:
    temp = cond_probs[token]
    # [0] = not-spam, [1] = spam — indices must match class counts
    temp[0] = (temp[0] * notspam_count + 1) / (notspam_count + V)
    temp[1] = (temp[1] * spam_count    + 1) / (spam_count    + V)
    cond_probs[token] = temp

# Verify smoothing worked — no zeros
print(f"P(blockchain | not-spam) = {cond_probs['blockchain'][0]:.6f}")
print(f"P(blockchain | spam)     = {cond_probs['blockchain'][1]:.6f}")
print(f"P(proposal   | not-spam) = {cond_probs['proposal'][0]:.6f}")
print(f"P(proposal   | spam)     = {cond_probs['proposal'][1]:.6f}")

P(blockchain | not-spam) = 0.015152
P(blockchain | spam)     = 0.045455
P(proposal   | not-spam) = 0.030303
P(proposal   | spam)     = 0.015152


## Step 8: Prediction with Smoothing

In [9]:
scores = list(class_priors)
for token in unlabelled_tokens:
    probs = cond_probs.get(token, [1/(notspam_count+V), 1/(spam_count+V)])
    scores[0] *= probs[0]
    scores[1] *= probs[1]

print(f"P(not-spam | '{unlabelled}') = {scores[0]:.8f}")
print(f"P(spam     | '{unlabelled}') = {scores[1]:.8f}")
print("\nPrediction:", "Spam" if scores[1] > scores[0] else "Not Spam")
print("\n'blockchain' is a strong spam signal; 'proposal' is more common in not-spam.")
print("Smoothing lets both signals contribute — correct classification restored.")

P(not-spam | 'blockchain proposal') = 0.00022957
P(spam     | 'blockchain proposal') = 0.00034435

Prediction: Spam

'blockchain' is a strong spam signal; 'proposal' is more common in not-spam.
Smoothing lets both signals contribute — correct classification restored.
